# Spectral analysis of a simulated trace

This notebook runs through the generation (or loading) of simulated data, and using that as the data as source to estimate spectral densities.

It simulates $10^6$ samples (0.1 μs steps, 100 ms) of a bead in a free-space tweezer with
`Simulation`, then estimates the position PSDs $S_{xx}$, $S_{yy}$ and the cross-spectral density
$S_{xy}$ with `SpectralDensityEstimate`, averaging 100 segments. **Runtime:** about a minute.

**Updated Nov 2026:** the simulation is seeded (`seed=0`), so it is reproducible; it is no longer
pickled next to the notebook by default (`SAVE_SIMULATION = True` writes it to `output/`, which is
not tracked); figures use `set_plt_rcParams_to_xmgrace_style()`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# can import from specific files
from levisim.core.colours import grey
from levisim.core.constants import π
from levisim.core.units import pm, kiloHz
from levisim.core.utils import load_variable_from_pickle

# can also directly import any of the variables in the entire levisim namespace
from levisim import Simulation, Experiment, SpectralDensityEstimate
from levisim.core.plotting import set_plt_rcParams_to_xmgrace_style

set_plt_rcParams_to_xmgrace_style()
# curves touch the left and right edges of the axes
plt.rcParams['axes.xmargin'] = 0

# seeded, so the time traces (and every number below) are reproducible
SEED = 0
# True also pickles the simulation into output/ (24 MB for 1e6 samples), for reloading below
SAVE_SIMULATION = False

if 0:
    # simulating 1e5 samples takes about 10s, and is 2.4MB
    sim = Simulation(num_samples=1e5, seed=SEED, save=SAVE_SIMULATION, save_dir='output')
elif 1:
    # simulating 1e6 samples takes about 90s = 1m30s, and is 24MB
    sim = Simulation(num_samples=1e6, seed=SEED, save=SAVE_SIMULATION, save_dir='output')
else:
    filename = 'default_P4e-03_fbx2e+03_t1e-07_N1e+05'
    filename = 'default_P4e-03_fbx2e+03_t1e-07_N1e+06'
    sim = load_variable_from_pickle(f'output/{filename}.pickle')

sim.info
sim.trap.info
sim.trap.tweezer.info
sim.trap.bead.info
sim.trap.gas.info

## Choosing how many segments to average

First, we must understand the basic relationships between the length and resolution of the data in the time domain, and how it determines the "length" (bandwidth) and "resolution" (frequency bin size) in the frequency domain:

- The length of the time trace $T$ deterimines the minimum possible frequency resolution
$$
\Delta f_{min} = \frac{1}{T}
$$

(Using the *whole* time trace for *one* estimate is known as *"one-shot"* estimate, i.e. it is done without any statistical averaging.)


- The time step of the time trace $\Delta t$ determines the maximum possible frequency that can be estimated, a.k.a the *Nyquist* frequency
$$
f_{Nyq} = \frac{1}{2 \Delta t}
$$

(The factor of $\frac{1}{2}$ accounts for the fact we only consider *positive* frequencies, i.e. *one-sided* spectral estimates. The full bandwidth of the spectral estimate, including negative frequencies, is $\frac{1}{\Delta t}$.)

- Since the error associated with a one-shot numerical estimate of a power spectral density is 100% [*"Random Data" by Bendat, Piersol; Eq. 9.30 p292*], it is desirable to do some statistical averaging to reduce this error.

- To do this, we divide the time trace into a *number of segments*, $n_{seg}$. The effect on the frequency resolution is simple:

$$
\Delta f = \frac{1}{T_{seg}}
$$
where $T_{seg}$ is the length of time of each segment, i.e.

$$
T_{seg} = \frac{T}{n_{seg}}
$$


In [ ]:
nseg = 100
Sxx = SpectralDensityEstimate(sim, 'xx', num_segments=nseg)
Syy = SpectralDensityEstimate(sim, 'yy', num_segments=nseg)
Sxy = SpectralDensityEstimate(sim, 'xy', num_segments=nseg)

# lower and upper bounds of the plots, in Hertz
lb, ub = 10e3, 250e3
ls = '.-'  # linestyle

# showing how to directly get at freqs, values from the Sxx, Syy objects
fig, ax = plt.subplots()
ax.plot(*Sxx.get_plot_points(lb, ub), ls, label='Sxx')
ax.plot(*Syy.get_plot_points(lb, ub), ls, label='Syy')
ax.set_title('PSDs')
ax.set_xlabel('Frequency (Hz)')
ax.set_ylabel('m²/Hz')
ax.legend()
ax.set_yscale('log')
plt.show()

# showing how to easily change axis units
freqs, csd = Sxy.get_plot_points(lb, ub)
freqs = freqs / kiloHz
csd = csd / pm**2

fig, ax = plt.subplots()
ax.plot(freqs, np.real(csd), ls, label='real')
ax.plot(freqs, np.imag(csd), ls, label='imag')
ax.plot(freqs, np.abs(csd), ls, label='abs')
ax.set_title('Parts of the CSD')
ax.set_xlabel('Frequency (kHz)')
ax.set_ylabel('pm²/Hz')
ax.legend()
plt.show()

fig, ax = plt.subplots()
ax.plot(freqs, np.angle(csd) / π, ls, color=grey, label='phase')
ax.set_title('Phase of the CSD')
ax.set_xlabel('Frequency (kHz)')
ax.set_ylabel('π radians')
ax.legend()
plt.show()

Sxx.info
Syy.info
Sxy.info